# 第四阶段：STL（标准模板库）

## 实验 3：`std::vector` —— 拥有连续元素

`std::vector<T>` 是拥有动态连续存储的 RAII 容器。本实验不只练习增删元素，而是研究三个与 Native SDK 直接相关的问题：

- `size()`、`capacity()` 和 `reserve()` 分别表达什么；
- 哪些操作会让 pointer、reference 和 iterator 失效；
- 如何把 vector 临时映射成 C ABI 常见的 pointer + length。

实验只读取失效前保存的地址数值，不会解引用已经失效的指针。

In [1]:
// 本步骤：引入本实验需要的标准库和公开头文件。
#include <algorithm>
#include <cassert>
#include <cstddef>
#include <cstdint>
#include <iostream>
#include <numeric>
#include <vector>

### 1. vector 拥有一段连续存储

vector 对象负责元素的构造、销毁以及底层存储的分配和释放。元素按索引连续排列，因此 `data() + i` 指向第 `i` 个元素。

```text
std::vector<T> owner
        │ owns
        ▼
┌─────┬─────┬─────┬──────────────┐
│ T 0 │ T 1 │ T 2 │ unused space │
└─────┴─────┴─────┴──────────────┘
      size = 3      capacity = 4
```

“连续”描述元素布局，“拥有”描述生命周期；`data()` 返回的指针只是借用，不转移所有权。

In [2]:
// 本步骤：通过代码演示“vector 拥有一段连续存储”并观察结果。
{
    // 创建拥有三个整数的 vector，作为连续存储的 owner。
    std::vector<int> values{10, 20, 30};

    // 输出元素数量和首地址，建立 size 与 data 的直观联系。
    std::cout << "size = " << values.size() << '\n';
    std::cout << "data = "
              << static_cast<const void *>(values.data())
              << '\n';

    // 逐个比较下标访问和 pointer arithmetic，验证元素连续排列。
    for (std::size_t index = 0; index < values.size(); ++index)
    {
        std::cout << "values[" << index << "] = "
                  << values[index]
                  << ", address = "
                  << static_cast<const void *>(values.data() + index)
                  << '\n';

        assert(values.data()[index] == values[index]);
    }
}

size = 3
data = 0x152e7d440
values[0] = 10, address = 0x152e7d440
values[1] = 20, address = 0x152e7d444
values[2] = 30, address = 0x152e7d448


### 2. 区分 `size()`、`capacity()`、`reserve()` 和 `resize()`

- `size()`：当前已经构造、可以访问的元素数量。
- `capacity()`：不重新分配存储时最多可容纳的元素数量。
- `reserve(n)`：只预留存储，不创建元素；标准只保证 capacity 至少为 `n`。
- `resize(n)`：真正改变元素数量，增长时会构造新元素。

capacity 的增长倍数属于实现细节，不要断言它必须翻倍。

In [3]:
// 本步骤：通过代码演示“区分 size()、capacity()、reserve() 和 resize()”并观察结果。
{
    // 先创建三个已构造元素，记录 reserve 前的逻辑大小。
    std::vector<int> values{10, 20, 30};

    const std::size_t size_before = values.size();
    // 只预留至少十个元素的存储，验证 reserve 不会创建元素。
    values.reserve(10);

    assert(values.size() == size_before);
    assert(values.capacity() >= 10);

    // 把 size 增长到五，并用 -1 构造新增的两个元素。
    values.resize(5, -1);

    // 输出并断言最终状态，区分逻辑元素数量与存储能力。
    std::cout << "size     = " << values.size() << '\n';
    std::cout << "capacity = " << values.capacity() << '\n';
    std::cout << "elements =";

    for (int value : values)
    {
        std::cout << ' ' << value;
    }

    std::cout << '\n';

    assert(values[3] == -1);
    assert(values[4] == -1);
}

size     = 5
capacity = 10
elements = 10 20 30 -1 -1


### 3. 复制 vector 得到独立的值

和 `std::string` 一样，vector 是 RAII 值类型。复制 vector 会复制其元素和存储；修改副本不会改变原对象。vector 本身析构时会依次销毁元素并释放存储，不需要手写 `delete[]`。

这不代表元素指向的外部对象也会被深复制。例如 `vector<T*>` 只复制指针值，并不接管或复制指针所指对象。

In [4]:
// 本步骤：通过代码演示“复制 vector 得到独立的值”并观察结果。
{
    // 复制 owner；copy 获得独立存储和独立元素。
    std::vector<int> original{10, 20, 30};
    std::vector<int> copy = original;

    // 只修改副本，用结果验证复制不是共享底层缓冲区。
    copy[0] = 99;
    copy.push_back(40);

    // 验证原对象未变，并输出两个 owner 各自的存储地址。
    assert(original == std::vector<int>({10, 20, 30}));
    assert(copy == std::vector<int>({99, 20, 30, 40}));

    std::cout << "original.data = "
              << static_cast<const void *>(original.data())
              << '\n';
    std::cout << "copy.data     = "
              << static_cast<const void *>(copy.data())
              << '\n';
}

original.data = 0x152e93550
copy.data     = 0x152ed5760


### 4. capacity 用尽时必然重分配

当插入后所需元素数量超过旧 capacity，vector 必须获得更大的连续存储，把已有元素转移过去，再释放旧存储：

```text
old storage ── move/copy elements ──> new larger storage
     │                                      │
     └──────── released                     └─ vector now owns this
```

旧存储中的 pointer、reference 和 iterator 此后全部失效。下面先把 vector 填满到 `size() == capacity()`，再插入一个元素，确保触发重分配。

In [5]:
// 本步骤：通过代码演示“capacity 用尽时必然重分配”并观察结果。
{
    // 预留一小段存储，再把 size 填充到恰好等于 capacity。
    std::vector<int> values;
    values.reserve(4);

    while (values.size() < values.capacity())
    {
        values.push_back(static_cast<int>(values.size() * 10));
    }

    // 在扩容前只保存容量和地址数值；失效后不会解引用旧 pointer。
    const std::size_t old_capacity = values.capacity();
    const std::uintptr_t old_address =
        reinterpret_cast<std::uintptr_t>(values.data());

    // 插入第 capacity + 1 个元素，确保 vector 必须重新分配。
    values.push_back(999);

    const std::uintptr_t new_address =
        reinterpret_cast<std::uintptr_t>(values.data());

    // 比较扩容前后的容量和地址，观察 owner 已切换到新存储。
    std::cout << "old capacity = " << old_capacity << '\n';
    std::cout << "new capacity = " << values.capacity() << '\n';
    std::cout << "old address  = " << old_address << '\n';
    std::cout << "new address  = " << new_address << '\n';
    std::cout << "reallocated  = "
              << std::boolalpha
              << (old_address != new_address)
              << '\n';

    // 用断言验证扩容后的容器状态以及地址确实发生变化。
    assert(values.size() == old_capacity + 1);
    assert(values.capacity() >= values.size());
    assert(old_address != new_address);
}

old capacity = 4
new capacity = 8
old address  = 5705943152
new address  = 5705830112
reallocated  = true


不能用旧指针“试试看是否还能读到原值”：

```cpp
const int *borrowed = values.data();
values.push_back(...); // 如果发生重分配，borrowed 失效
std::cout << *borrowed; // undefined behavior
```

地址看起来没有变化也不能作为通用安全证明；必须根据操作的失效规则判断。失效后，旧地址只适合用于比较和日志，不再代表可访问对象。

### 5. 没有重分配也要检查具体操作

提前 `reserve()` 可以减少扩容，但不能让所有 iterator 永久有效。若 `push_back()` 没有触发重分配，已有元素的 pointer/reference/iterator 保持有效，旧的 past-the-end iterator 仍会失效。

In [6]:
// 本步骤：通过代码演示“没有重分配也要检查具体操作”并观察结果。
{
    // 预留足够容量，使后续 push_back 不触发重分配。
    std::vector<int> values;
    values.reserve(8);
    values.push_back(10);
    values.push_back(20);

    // 在修改前借用首元素，并记录当前存储地址。
    int *first = &values.front();
    const std::uintptr_t address_before =
        reinterpret_cast<std::uintptr_t>(values.data());

    // 在剩余 capacity 内增加元素，已有元素地址应保持不变。
    values.push_back(30);

    const std::uintptr_t address_after =
        reinterpret_cast<std::uintptr_t>(values.data());

    // 验证旧借用仍指向首元素，并输出本次没有重分配的证据。
    assert(address_before == address_after);
    assert(first == &values.front());
    assert(*first == 10);

    std::cout << "storage unchanged = "
              << std::boolalpha
              << (address_before == address_after)
              << '\n';
}

storage unchanged = true


常见规则可先记住以下几条：

| 操作 | 未重分配时 | 发生重分配时 |
| --- | --- | --- |
| `push_back` / `emplace_back` | 旧 `end()` 失效 | 全部 pointer/reference/iterator 失效 |
| `insert` | 插入点及其后的借用失效 | 全部失效 |
| `erase` | 删除点及其后的借用失效 | 不适用 |
| `reserve` | capacity 未变化则不失效 | 全部失效 |
| `clear` | 所有元素借用失效 | capacity 通常保留 |

`shrink_to_fit()` 只是非强制请求，不能依赖它一定缩容。跨 API 边界前应先完成所有可能改变 vector 结构的操作。

### 6. iterator 把 vector 接入标准算法

vector 不只提供下标访问。它的 iterator 表达一个元素范围，可以直接交给 `sort`、`find`、`accumulate` 等算法。范围通常写成半开区间 `[begin, end)`：包含 begin，不包含 end。

In [7]:
// 本步骤：通过代码演示“iterator 把 vector 接入标准算法”并观察结果。
{
    // 创建无序输入，准备通过 iterator 范围交给标准算法。
    std::vector<int> values{40, 10, 30, 20};

    // 原地排序整个半开区间 [begin, end)。
    std::sort(values.begin(), values.end());

    // 使用只读 iterator 查找 30，并聚合全部元素。
    const auto found = std::find(
        values.cbegin(),
        values.cend(),
        30);

    const int sum = std::accumulate(
        values.cbegin(),
        values.cend(),
        0);

    // 验证排序、查找和求和结果，再输出可观察结果。
    assert(values == std::vector<int>({10, 20, 30, 40}));
    assert(found != values.cend());
    assert(*found == 30);
    assert(sum == 100);

    std::cout << "sorted =";

    for (int value : values)
    {
        std::cout << ' ' << value;
    }

    std::cout << "\nsum = " << sum << '\n';
}

sorted = 10 20 30 40
sum = 100


### 7. 在 C ABI 边界借用 pointer + length

C ABI 不能直接暴露 `std::vector`：它是 C++ 类型，其布局、模板实例和异常行为都不是稳定的 C 契约。输入缓冲区通常拆成 `const T *data` 与 `size_t size`。C++ 调用方可在一次同步调用内传入 `vector.data()` 和 `vector.size()`。

In [ ]:
// 本步骤：通过代码演示“在 C ABI 边界借用 pointer + length”并观察结果。
std::uint64_t sdk_checksum(
    const std::uint8_t *data,
    std::size_t size)
{
    // 先验证 C ABI 的 pointer + length 契约，空输入允许空指针。
    assert(data != nullptr || size == 0);

    // 只在 size 指定的边界内读取借用缓冲区并计算校验和。
    std::uint64_t result = 0;

    for (std::size_t index = 0; index < size; ++index)
    {
        result += data[index];
    }

    return result;
}

{
    // 由 vector 持有 payload，并在同步调用期间临时借出存储。
    std::vector<std::uint8_t> payload{10, 20, 30, 40};

    const std::uint64_t checksum =
        sdk_checksum(payload.data(), payload.size());

    // 输出并验证 native 函数只读借用得到的计算结果。
    std::cout << "checksum = " << checksum << '\n';
    assert(checksum == 100);
}

这次转换没有复制，也没有转移所有权：

```text
std::vector<uint8_t> owner
        │ data() + size()
        ▼
C ABI synchronous call
        │ borrows during call only
        ▼
return ── borrow ends
```

调用期间，vector 必须仍然存活且不能执行可能让存储失效的修改。若 Native SDK 要在函数返回后、其他线程或异步 callback 中继续使用数据，必须复制内容，或设计明确的 retain/release、create/destroy 协议。

Kotlin/Native 的 `ByteArray` 也不能直接当作长期稳定的 native 指针。它只能在受约束的 pinned/作用域借用期间传入 pointer + length；native 端不得私自保存该指针。

### 8. 类型与接口边界

跨 ABI 时还要检查元素类型：

- `std::vector<int>` 中 `int` 的宽度不应被跨平台协议默认为固定值；二进制接口优先使用 `std::uint8_t`、`std::int32_t` 等明确宽度类型。
- 长度使用元素数量，不要把 `size()` 错当成字节数；需要字节数时应明确计算 `size() * sizeof(T)`。
- `std::vector<bool>` 是特殊的位压缩容器，不提供普通 `bool *` 元素模型，不适合直接映射为 C 缓冲区。
- 空 vector 的 `data()` 可能是空指针，也可能是不可解引用的非空值；只要 size 为 0，callee 就不能解引用它。

如果 API 需要把输出交给调用方长期持有，不要返回局部 vector 的 `data()`。应复制到调用方缓冲区，或返回带有配套释放函数的不透明 handle。

### 本实验结论

`std::vector<T>` 是拥有连续元素存储的 RAII 值类型。`size()` 描述已构造元素，`capacity()` 描述当前存储能力，`reserve()` 只预留空间，`resize()` 才改变元素数量。

`data()`、reference 和 iterator 都是对内部元素的借用。判断它们是否有效，不能依赖一次运行看到的地址，而要检查 owner 是否存活、容器操作是否触发重分配，以及该操作自身的失效规则。

跨 C ABI 时只传递明确元素类型的 pointer + length，并把借用限制在同步调用期间。进入异步、跨线程或长期保存场景时，应复制数据或升级为明确的所有权协议。